In [1]:
from pathlib import Path
import sys
project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path('/data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR')
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Experimentation with TSFEDL Time-Series Models

This notebook mirrors the experiment-oriented structure of `experiment_pyod_models.ipynb`, but using TSFEDL-based time-series models integrated in RADAR.

It uses the same two UCI datasets already explored in the time-series examples and benchmarks two TSFEDL architectures in anomaly-detection mode through reconstruction error.

## Import Required Libraries

Imports for data loading, preprocessing, window creation, TSFEDL models, metrics, and export of the experiment tables.

In [2]:
import importlib
import time
from inspect import signature
from pathlib import Path
from statistics import mean

import numpy as np
import pandas as pd
import pytorch_lightning as pl
import torch
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

from TSFEDL.models_pytorch import (
    CaiWenjuan_Forecaster,
    ChenChen_Forecaster,
    DaiXiLi_Forecaster,
    FuJiangmeng_Forecaster,
    GaoJunLi_Forecaster,
    GenMinxing_Forecaster,
    HongTan_Forecaster,
    HtetMyetLynn_Forecaster,
    HuangMeiLing_Forecaster,
    KhanZulfiqar_Forecaster,
    KimTaeYoung_Forecaster,
    KongZhengmin_Forecaster,
    LihOhShu_Forecaster,
    OhShuLih_Forecaster,
    SharPar_Forecaster,
    ShiHaotian_Forecaster,
    WangKejun_Forecaster,
    WeiXiaoyan_Forecaster,
    YaoQihang_Forecaster,
    YiboGao_Forecaster,
    YildirimOzal_Forecaster,
    ZhangJin_Forecaster,
    ZhengZhenyu_Forecaster,
)
from RADAR.time_series.algorithms import tsfedl
from RADAR.time_series.preprocessing.preprocessing_ts import StandardScalerPreprocessing
from RADAR.time_series.time_series_datasets_uci import global_load as load_time_series
from RADAR.time_series.time_series_utils import TimeSeriesProcessor
import RADAR.metrics_module as metrics_module

metrics_module = importlib.reload(metrics_module)

2026-03-13 21:45:17.162442: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-13 21:45:17.176018: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773434717.192496  356609 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773434717.196402  356609 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-13 21:45:17.212837: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

## Build the UCI Time-Series Benchmark

The preprocessing follows the same dataset choices as `test_tsfedl.ipynb` and keeps a chronological split.

- `ai4i_2020_predictive_maintenance_dataset`: labels come from `Machine failure`.
- `metro_interstate_traffic_volume`: anomaly labels are derived from extreme traffic levels using the 5th and 95th percentiles.

To keep the benchmark aligned with anomaly detection, the TSFEDL models are trained in reconstruction mode, using each input window as its own target.

In [4]:
WINDOW_SIZE = 48
STEP_SIZE = 1
TEST_SIZE = 0.2
METRO_LOW_Q = 0.05
METRO_HIGH_Q = 0.95

def chronological_split(X, y, test_size=0.2):
    split_idx = int(len(X) * (1 - test_size))
    return X[:split_idx], X[split_idx:], y[:split_idx], y[split_idx:]

def aggregate_window_labels(y_windows):
    y_windows = np.asarray(y_windows)
    if y_windows.ndim == 1:
        return y_windows.astype(int)
    return (y_windows.sum(axis=1) > 0).astype(int)

def prepare_ai4i_dataset(window_size=WINDOW_SIZE, step_size=STEP_SIZE, test_size=TEST_SIZE):
    X, y = load_time_series('ai4i_2020_predictive_maintenance_dataset')
    labels = y['Machine failure'].astype(int).to_numpy()
    X = X.drop(columns=['Type'], errors='ignore')

    scaler = StandardScalerPreprocessing()
    X_scaled = scaler.fit_transform(X)
    X_values = np.asarray(X_scaled, dtype=np.float32)

    X_train, X_test, y_train, y_test = chronological_split(X_values, labels, test_size=test_size)

    processor = TimeSeriesProcessor(window_size=window_size, step_size=step_size, future_prediction=False)
    X_train_windows, y_train_windows, X_test_windows, y_test_windows = processor.process_train_test(X_train, y_train, X_test, y_test)

    return {
        'dataset': 'ai4i_2020_predictive_maintenance_dataset',
        'X_train_windows': np.asarray(X_train_windows, dtype=np.float32),
        'X_test_windows': np.asarray(X_test_windows, dtype=np.float32),
        'y_test_labels': aggregate_window_labels(y_test_windows),
        'n_samples': len(X_values),
        'n_features': X_values.shape[1],
        'window_size': window_size,
        'train_windows': len(X_train_windows),
        'test_windows': len(X_test_windows),
        'positive_ratio_points': round(float(np.mean(labels)), 4),
        'positive_ratio_windows': round(float(np.mean(aggregate_window_labels(y_test_windows))), 4),
        'label_note': 'Machine failure from UCI target',
    }

def prepare_metro_dataset(window_size=WINDOW_SIZE, step_size=STEP_SIZE, test_size=TEST_SIZE, low_q=METRO_LOW_Q, high_q=METRO_HIGH_Q):
    X, y = load_time_series('metro_interstate_traffic_volume')
    traffic_volume = y['traffic_volume'].astype(float)
    low_threshold = float(traffic_volume.quantile(low_q))
    high_threshold = float(traffic_volume.quantile(high_q))
    labels = ((traffic_volume <= low_threshold) | (traffic_volume >= high_threshold)).astype(int).to_numpy()

    X = X.drop(columns=['date_time', 'holiday', 'weather_main', 'weather_description'], errors='ignore')

    scaler = StandardScalerPreprocessing()
    X_scaled = scaler.fit_transform(X)
    X_values = np.asarray(X_scaled, dtype=np.float32)

    X_train, X_test, y_train, y_test = chronological_split(X_values, labels, test_size=test_size)

    processor = TimeSeriesProcessor(window_size=window_size, step_size=step_size, future_prediction=False)
    X_train_windows, y_train_windows, X_test_windows, y_test_windows = processor.process_train_test(X_train, y_train, X_test, y_test)

    return {
        'dataset': 'metro_interstate_traffic_volume',
        'X_train_windows': np.asarray(X_train_windows, dtype=np.float32),
        'X_test_windows': np.asarray(X_test_windows, dtype=np.float32),
        'y_test_labels': aggregate_window_labels(y_test_windows),
        'n_samples': len(X_values),
        'n_features': X_values.shape[1],
        'window_size': window_size,
        'train_windows': len(X_train_windows),
        'test_windows': len(X_test_windows),
        'positive_ratio_points': round(float(np.mean(labels)), 4),
        'positive_ratio_windows': round(float(np.mean(aggregate_window_labels(y_test_windows))), 4),
        'label_note': f'Extreme traffic volume: <= q{low_q:.2f} or >= q{high_q:.2f}',
        'low_threshold': round(low_threshold, 3),
        'high_threshold': round(high_threshold, 3),
    }

dataset_configs = {
    'ai4i': prepare_ai4i_dataset(),
    'metro_interstate': prepare_metro_dataset(),
}

Metadata: {'uci_id': 601, 'name': 'AI4I 2020 Predictive Maintenance Dataset', 'repository_url': 'https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset', 'data_url': 'https://archive.ics.uci.edu/static/public/601/data.csv', 'abstract': 'The AI4I 2020 Predictive Maintenance Dataset is a synthetic dataset that reflects real predictive maintenance data encountered in industry.', 'area': 'Computer Science', 'tasks': ['Classification', 'Regression', 'Causal-Discovery'], 'characteristics': ['Multivariate', 'Time-Series'], 'num_instances': 10000, 'num_features': 6, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'], 'index_col': ['UID', 'Product ID'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2020, 'last_updated': 'Wed Feb 14 2024', 'dataset_doi': '10.24432/C5HS5C', 'creators': [], 'intro_paper': {'ID': 386, 'type': 'NATIVE', 'title': 'Explainable Artificial 

In [5]:
dataset_summary = pd.DataFrame([
    {
        'dataset_key': dataset_key,
        'dataset_name': config['dataset'],
        'samples': config['n_samples'],
        'features': config['n_features'],
        'window_size': config['window_size'],
        'train_windows': config['train_windows'],
        'test_windows': config['test_windows'],
        'positive_ratio_points': config['positive_ratio_points'],
        'positive_ratio_windows': config['positive_ratio_windows'],
        'label_note': config['label_note'],
    }
    for dataset_key, config in dataset_configs.items()
]).reset_index(drop=True)

display(dataset_summary)

,dataset_key,dataset_name,samples,features,window_size,train_windows,test_windows,positive_ratio_points,positive_ratio_windows,label_note
0,ai4i,ai4i_2020_predictive_maintenance_dataset,10000,5,48,7953,1953,0.0339,0.5832,Machine failure from UCI target
1,metro_interstate,metro_interstate_traffic_volume,48204,4,48,38516,9594,0.1006,0.8919,Extreme traffic volume: <= q0.05 or >= q0.95


## TSFEDL Benchmark on the Two UCI Datasets


In [14]:
def get_forecaster_configs(input_dim):
    return {
        'ohshulih': {
            'class': OhShuLih_Forecaster,
            'display_name': 'OhShuLih',
            'extra_kwargs': {},
            'top_module_kwargs': {},
        },
        'gaojunli': {
            'class': GaoJunLi_Forecaster,
            'display_name': 'GaoJunLi',
            'extra_kwargs': {},
            'top_module_kwargs': {},
        },
        'kongzhengmin': {
            'class': KongZhengmin_Forecaster,
            'display_name': 'KongZhengmin',
            'extra_kwargs': {},
            'top_module_kwargs': {},
        },
        'caiwenjuan': {
            'class': CaiWenjuan_Forecaster,
            'display_name': 'CaiWenjuan',
            'extra_kwargs': {},
            'top_module_kwargs': {},
        },
        'wangkejun': {
            'class': WangKejun_Forecaster,
            'display_name': 'WangKejun',
            'extra_kwargs': {},
            'top_module_kwargs': {},
        },
        'zhengzhenyu': {
            'class': ZhengZhenyu_Forecaster,
            'display_name': 'ZhengZhenyu',
            'extra_kwargs': {},
            'top_module_kwargs': {'in_features': 256},
        },
        'kimtaeyoung': {
            'class': KimTaeYoung_Forecaster,
            'display_name': 'KimTaeYoung',
            'extra_kwargs': {},
            'top_module_kwargs': {},
        },
        'fujiangmeng': {
            'class': FuJiangmeng_Forecaster,
            'display_name': 'FuJiangmeng',
            'extra_kwargs': {},
            'top_module_kwargs': {},
        },
        'shihaotian': {
            'class': ShiHaotian_Forecaster,
            'display_name': 'ShiHaotian',
            'extra_kwargs': {},
            'top_module_kwargs': {},
        },
        'sharpar': {
            'class': SharPar_Forecaster,
            'display_name': 'SharPar',
            'extra_kwargs': {},
            'top_module_kwargs': {},
        },
        'hongtan': {
            'class': HongTan_Forecaster,
            'display_name': 'HongTan',
            'extra_kwargs': {},
            'top_module_kwargs': {},
        },
        'htetmyetlynn': {
            'class': HtetMyetLynn_Forecaster,
            'display_name': 'HtetMyetLynn',
            'extra_kwargs': {},
            'top_module_kwargs': {},
        },
        'liohshu': {
            'class': LihOhShu_Forecaster,
            'display_name': 'LihOhShu',
            'extra_kwargs': {},
            'top_module_kwargs': {},
        },
        'yibogao': {
            'class': YiboGao_Forecaster,
            'display_name': 'YiboGao',
            'extra_kwargs': {},
            'top_module_kwargs': {},
        },
        'yaoqihang': {
            'class': YaoQihang_Forecaster,
            'display_name': 'YaoQihang',
            'extra_kwargs': {},
            'top_module_kwargs': {},
        },
        'yildirimozal': {
            'class': YildirimOzal_Forecaster,
            'display_name': 'YildirimOzal',
            'extra_kwargs': {},
            'top_module_kwargs': {'input_shape': (126, 126), 'in_features': input_dim},
        },
        'zhangjin': {
            'class': ZhangJin_Forecaster,
            'display_name': 'ZhangJin',
            'extra_kwargs': {},
            'top_module_kwargs': {'in_features': input_dim},
        },
        'weixiaoyan': {
            'class': WeiXiaoyan_Forecaster,
            'display_name': 'WeiXiaoyan',
            'extra_kwargs': {},
            'top_module_kwargs': {},
        },
        'khanzulfiqar': {
            'class': KhanZulfiqar_Forecaster,
            'display_name': 'KhanZulfiqar',
            'extra_kwargs': {},
            'top_module_kwargs': {},
        },
        'chenchen': {
            'class': ChenChen_Forecaster,
            'display_name': 'ChenChen',
            'extra_kwargs': {},
            'top_module_kwargs': {},
        },
        'genminxing': {
            'class': GenMinxing_Forecaster,
            'display_name': 'GenMinxing',
            'extra_kwargs': {},
            'top_module_kwargs': {},
        },
        'huangmeiling': {
            'class': HuangMeiLing_Forecaster,
            'display_name': 'HuangMeiLing',
            'extra_kwargs': {},
            'top_module_kwargs': {},
        },
        'daixili': {
            'class': DaiXiLi_Forecaster,
            'display_name': 'DaiXiLi',
            'extra_kwargs': {},
            'top_module_kwargs': {},
        },
    }

In [7]:
# List of forecasters to benchmark, in desired order
ordered_forecasters = [
    'ohshulih', 'gaojunli', 'kongzhengmin', 'caiwenjuan', 'wangkejun', 'zhengzhenyu',
    'kimtaeyoung', 'fujiangmeng', 'shihaotian', 'sharpar', 'hongtan', 'htetmyetlynn',
    'liohshu', 'yibogao', 'yaoqihang', 'yildirimozal', 'zhangjin', 'weixiaoyan',
    'khanzulfiqar', 'chenchen', 'genminxing', 'huangmeiling', 'daixili'
]


In [8]:
# Number of times to repeat timing for each model (for averaging)
TIMING_REPETITIONS = 1  # You can increase this for more robust timing, e.g., 3 or 5


In [15]:
BATCH_SIZE = 64
MAX_EPOCHS = 1
TRAINER_DEFAULT_KWARGS = {
    'max_epochs': MAX_EPOCHS,
    'logger': False,
    'enable_checkpointing': False,
    'enable_progress_bar': False,
}


def summarize_mse(scores):
    scores = np.asarray(scores, dtype=float).ravel()
    finite_scores = scores[np.isfinite(scores)]
    return float(np.mean(finite_scores)) if finite_scores.size else np.nan


def build_model_params(forecaster_name, input_dim, seq_len):
    forecaster_config = get_forecaster_configs(input_dim)[forecaster_name]
    top_module_kwargs = {
        'out_features': input_dim,
        'n_pred': seq_len,
        **forecaster_config.get('top_module_kwargs', {}),
    }
    top_module = forecaster_config['class'](**top_module_kwargs)

    return {
        'algorithm_': forecaster_name,
        'loss': torch.nn.MSELoss(),
        'top_module': top_module,
        'batch_size': BATCH_SIZE,
        'in_features': seq_len,
        **TRAINER_DEFAULT_KWARGS,
        **forecaster_config.get('extra_kwargs', {}),
    }


def split_model_and_trainer_params(model_params):
    trainer_signature = signature(pl.Trainer.__init__)
    trainer_param_names = {name for name in trainer_signature.parameters if name != 'self'}

    trainer_kwargs = {}
    direct_model_kwargs = {}
    for key, value in model_params.items():
        if key in trainer_param_names:
            trainer_kwargs[key] = value
        else:
            direct_model_kwargs[key] = value

    return direct_model_kwargs, trainer_kwargs


def build_direct_tsfedl_model(model_kwargs):
    direct_model = model_kwargs['top_module']
    if hasattr(direct_model, 'loss'):
        direct_model.loss = model_kwargs['loss']
    return direct_model


def fit_direct_tsfedl_model(direct_model, X_train_tensor, batch_size, trainer_kwargs):
    train_dataset = TensorDataset(X_train_tensor, X_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    trainer = pl.Trainer(**trainer_kwargs)
    trainer.fit(direct_model, train_dataloaders=train_loader)
    return direct_model


def compute_direct_scores(direct_model, X_test_tensor):
    direct_model.eval()
    device = next(direct_model.parameters()).device if any(True for _ in direct_model.parameters()) else torch.device('cpu')
    X_test_tensor = X_test_tensor.to(device)

    with torch.no_grad():
        predictions = direct_model(X_test_tensor)
        if isinstance(predictions, (tuple, list)):
            predictions = predictions[0]

        if predictions.shape == X_test_tensor.shape:
            errors = (predictions - X_test_tensor) ** 2
            reduction_dims = tuple(range(1, errors.ndim))
            return torch.mean(errors, dim=reduction_dims).detach().cpu().numpy()

        predictions_flat = predictions.reshape(predictions.shape[0], -1)
        targets_flat = X_test_tensor.reshape(X_test_tensor.shape[0], -1)
        min_width = min(predictions_flat.shape[1], targets_flat.shape[1])
        errors = (predictions_flat[:, :min_width] - targets_flat[:, :min_width]) ** 2
        return torch.mean(errors, dim=1).detach().cpu().numpy()


In [ ]:
# ...definiciones previas...

tsfedl_results = []

for dataset_key, config in dataset_configs.items():
    input_dim = config['X_train_windows'].shape[2]
    seq_len = config['window_size']
    X_train_tensor = torch.tensor(config['X_train_windows'], dtype=torch.float32)
    X_test_tensor = torch.tensor(config['X_test_windows'], dtype=torch.float32)

    # Usar configuración dinámica de modelos según input_dim
    FORECASTER_CONFIGS = get_forecaster_configs(input_dim)

    print(f'\nDataset: {config["dataset"]}')
    print(f'Features: {input_dim} | Train windows: {config["train_windows"]} | Test windows: {config["test_windows"]}')

    for model_order, forecaster_name in enumerate(ordered_forecasters, start=1):
        display_name = FORECASTER_CONFIGS[forecaster_name]['display_name']
        print(f'  [{model_order:02d}] Model: {display_name}')

        result_row = {
            'dataset_key': dataset_key,
            'dataset_name': config['dataset'],
            'model_order': model_order,
            'model_display_name': display_name,
            'algorithm': forecaster_name,
            'window_size': seq_len,
            'n_features': input_dim,
            'train_windows': config['train_windows'],
            'test_windows': config['test_windows'],
            'timing_repetitions': TIMING_REPETITIONS,
            'platform_status': 'pending',
            'base_status': 'pending',
            'platform_error_message': None,
            'base_error_message': None,
            'average_platform_time_s': np.nan,
            'average_base_time_s': np.nan,
            'overhead_s': np.nan,
            'speedup_base_over_platform': np.nan,
            'platform_mse': np.nan,
            'base_mse': np.nan,
            'mse_diff': np.nan,
        }

        try:
            platform_execution_times = []
            platform_model = None
            for _ in tqdm(
                range(TIMING_REPETITIONS),
                desc=f'Platform Timing ({display_name} | {dataset_key})',
                leave=False,
            ):
                platform_params = build_model_params(forecaster_name, input_dim, seq_len)
                platform_model = tsfedl.TsfedlAnomalyDetection(**platform_params)
                start_time = time.time()
                platform_model.fit(config['X_train_windows'], X_train_tensor)
                platform_execution_times.append(time.time() - start_time)

            average_platform_time = mean(platform_execution_times)
            platform_scores = np.asarray(platform_model.decision_function(config['X_test_windows'])).ravel()
            finite_platform_scores = bool(np.isfinite(platform_scores).all())

            result_row.update({
                'platform_status': 'ok',
                'average_platform_time_s': round(average_platform_time, 4),
                'platform_mse': round(summarize_mse(platform_scores), 6) if finite_platform_scores else np.nan,
            })

            print(
                f'    Platform -> time={average_platform_time:.4f}s | '
                f'MSE={result_row["platform_mse"]:.6f}'
            )
        except Exception as exc:
            result_row['platform_status'] = 'failed'
            result_row['platform_error_message'] = str(exc).strip() or exc.__class__.__name__
            print(f'    Platform failed: {result_row["platform_error_message"]}')

        try:
            direct_execution_times = []
            direct_model = None
            for _ in tqdm(
                range(TIMING_REPETITIONS),
                desc=f'Direct TSFEDL Timing ({display_name} | {dataset_key})',
                leave=False,
            ):
                platform_params = build_model_params(forecaster_name, input_dim, seq_len)
                model_kwargs, trainer_kwargs = split_model_and_trainer_params(platform_params)
                direct_model = build_direct_tsfedl_model(model_kwargs)
                start_time = time.time()
                fit_direct_tsfedl_model(
                    direct_model=direct_model,
                    X_train_tensor=X_train_tensor,
                    batch_size=platform_params['batch_size'],
                    trainer_kwargs=trainer_kwargs,
                )
                direct_execution_times.append(time.time() - start_time)

            average_base_time = mean(direct_execution_times)
            direct_scores = compute_direct_scores(direct_model, X_test_tensor)
            finite_direct_scores = bool(np.isfinite(direct_scores).all())

            result_row.update({
                'base_status': 'ok',
                'average_base_time_s': round(average_base_time, 4),
                'base_mse': round(summarize_mse(direct_scores), 6) if finite_direct_scores else np.nan,
            })

            print(f'    Direct TSFEDL -> time={average_base_time:.4f}s | MSE={result_row["base_mse"]:.6f}')
        except Exception as exc:
            result_row['base_status'] = 'failed'
            result_row['base_error_message'] = str(exc).strip() or exc.__class__.__name__
            print(f'    Direct TSFEDL failed: {result_row["base_error_message"]}')

        if np.isfinite(result_row['average_platform_time_s']) and np.isfinite(result_row['average_base_time_s']):
            result_row['overhead_s'] = round(
                result_row['average_platform_time_s'] - result_row['average_base_time_s'],
                4,
            )
            result_row['speedup_base_over_platform'] = round(
                result_row['average_base_time_s'] / result_row['average_platform_time_s'],
                4,
            ) if result_row['average_platform_time_s'] > 0 else np.nan

        if np.isfinite(result_row['platform_mse']) and np.isfinite(result_row['base_mse']):
            result_row['mse_diff'] = round(
                result_row['platform_mse'] - result_row['base_mse'],
                6,
            )

        tsfedl_results.append(result_row)

tsfedl_results_df = pd.DataFrame(tsfedl_results).sort_values(
    ['dataset_name', 'model_order'],
    ascending=[True, True],
).reset_index(drop=True)

display(tsfedl_results_df)

## Per-Dataset Summary

This table compares the RADAR execution path with the direct TSFEDL library execution path and highlights the resulting overhead together with the reconstruction error reported as MSE.

In [ ]:
tsfedl_summary_df = tsfedl_results_df[
    [
        'dataset_name',
        'model_order',
        'model_display_name',
        'algorithm',
        'platform_status',
        'base_status',
        'average_platform_time_s',
        'average_base_time_s',
        'overhead_s',
        'speedup_base_over_platform',
        'platform_mse',
        'base_mse',
        'mse_diff',
        'platform_error_message',
        'base_error_message',
    ]
] .copy()

display(tsfedl_summary_df)

,dataset_name,model_order,model_display_name,algorithm,platform_status,base_status,average_platform_time_s,average_base_time_s,overhead_s,speedup_base_over_platform,platform_mse,base_mse,mse_diff,platform_error_message,base_error_message
0,ai4i_2020_predictive_maintenance_dataset,1,OhShuLih,ohshulih,ok,ok,4.6105,4.1694,0.4411,0.9043,0.462775,0.464380,-0.001605,None,None
1,ai4i_2020_predictive_maintenance_dataset,2,GaoJunLi,gaojunli,ok,ok,2.9106,2.8733,0.0373,0.9872,0.464318,0.462828,0.001490,None,None
2,ai4i_2020_predictive_maintenance_dataset,3,KongZhengmin,kongzhengmin,failed,failed,NaN,NaN,NaN,NaN,NaN,NaN,NaN,max_pool1d() Invalid computed output size: 0,max_pool1d() Invalid computed output size: 0
3,ai4i_2020_predictive_maintenance_dataset,4,CaiWenjuan,caiwenjuan,failed,failed,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Given input size: (43x1x1). Calculated output ...,Given input size: (43x1x1). Calculated output ...
4,ai4i_2020_predictive_maintenance_dataset,5,WangKejun,wangkejun,failed,failed,NaN,NaN,NaN,NaN,NaN,NaN,NaN,max_pool1d() Invalid computed output size: 0,max_pool1d() Invalid computed output size: 0
5,ai4i_2020_predictive_maintenance_dataset,6,ZhengZhenyu,zhengzhenyu,failed,failed,NaN,NaN,NaN,NaN,NaN,NaN,NaN,max_pool1d() Invalid computed output size: 0,max_pool1d() Invalid computed output size: 0
6,ai4i_2020_predictive_maintenance_dataset,7,KimTaeYoung,kimtaeyoung,ok,ok,3.9499,3.9820,-0.0321,1.0081,0.448110,0.446796,0.001314,None,None
7,ai4i_2020_predictive_maintenance_dataset,8,FuJiangmeng,fujiangmeng,ok,ok,5.3031,5.4609,-0.1578,1.0298,0.450814,0.451868,-0.001054,None,None
8,ai4i_2020_predictive_maintenance_dataset,9,ShiHaotian,shihaotian,failed,failed,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Calculated padded input size per channel: (5)....,Calculated padded input size per channel: (5)....
9,ai4i_2020_predictive_maintenance_dataset,10,SharPar,sharpar,ok,ok,4.9986,4.9119,0.0867,0.9827,0.443369,0.444795,-0.001426,None,None


In [ ]:
results_dir = project_root / 'results'
results_dir.mkdir(parents=True, exist_ok=True)

results_main_path = results_dir / 'uci_tsfedl_results.csv'
results_summary_path = results_dir / 'uci_tsfedl_summary.csv'

tsfedl_results_df.to_csv(results_main_path, index=False)
tsfedl_summary_df.to_csv(results_summary_path, index=False)

print(f'Saved detailed results to: {results_main_path}')
print(f'Saved summary results to: {results_summary_path}')

Saved detailed results to: /data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/results/uci_tsfedl_results.csv
Saved summary results to: /data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/results/uci_tsfedl_summary.csv
